# Aula Prática — Schema e Evolução de Dados no Kafka
### Computação Paralela e Distribuída (DCE540)

Até agora, publicamos e lemos mensagens no Kafka como **JSON solto** — qualquer producer podia enviar qualquer estrutura, e qualquer consumer simplesmente esperava encontrar certos campos, sem nenhuma validação. Isso funciona numa aula, mas não escala em produção:

- Nenhuma validação de estrutura (um producer pode enviar um tipo errado num campo, e isso só explode no consumer, silenciosamente);
- Evolução de schema é um campo minado (adicionar, renomear ou mudar o tipo de um campo pode quebrar consumidores antigos, sem aviso);
- JSON é texto, verboso — formatos binários como Avro são bem mais compactos.

Nesta aula vamos resolver isso com **Avro + Schema Registry**, simulando um cenário realista com **múltiplos produtores e múltiplos consumidores**, cada um evoluindo de forma independente — exatamente a situação em que um Schema Registry se torna indispensável.

**Pré-requisito:** aulas anteriores de Kafka (tópicos, partições, consumer group, semânticas de entrega).

**Importante sobre a biblioteca usada:** vamos manter tudo em `kafka-python` (como nas aulas anteriores) e implementar a comunicação com o Schema Registry **manualmente**, via chamadas HTTP simples (`requests`) e a biblioteca `fastavro` para codificar/decodificar os dados Avro. Isso é mais verboso do que usar a biblioteca oficial `confluent-kafka` (que tem um cliente de Schema Registry embutido), mas tem a vantagem de expor exatamente o que está acontecendo por baixo dos panos — o que é ótimo para aprender o mecanismo antes de usar a versão "de fábrica". A Seção 7, ao final, mostra como isso ficaria com `confluent-kafka`.


## 0. Preparando o ambiente (VS Code + Docker)

Esta aula usa um ambiente **novo e independente** das anteriores, com um serviço adicional: o **Schema Registry**. Siga estes passos:

1. Confirme que o **VS Code** tem as extensões **Python** e **Jupyter** instaladas.
2. Confirme que o **Docker Desktop** está aberto e rodando.
3. Se os containers de uma aula anterior ainda estiverem rodando, pare-os primeiro (esta aula usa as portas 9092, 9094 e 8080, iguais às anteriores):

   ```bash
   # na pasta da aula anterior que ainda estiver de pé
   docker compose down
   ```

4. Abra a pasta **desta aula** no VS Code — `docker-compose.yml`, este notebook e `requirements.txt` devem estar juntos na mesma pasta.
5. Em um terminal do VS Code, crie o ambiente virtual e instale as dependências:

   **Windows (PowerShell):**
   ```powershell
   python -m venv venv
   venv\Scripts\activate
   python -m pip install --upgrade pip
   python -m pip install -r requirements.txt
   ```

   **Linux/Mac:**
   ```bash
   python3 -m venv venv
   source venv/bin/activate
   python3 -m pip install --upgrade pip
   python3 -m pip install -r requirements.txt
   ```

   > **Use sempre `python -m pip install ...`, nunca `pip install ...` sozinho** (veja a nota na aula anterior sobre pastas no OneDrive/Google Drive causando instalação no lugar errado).

6. Abra o notebook desta aula no VS Code e selecione o kernel `venv`.
7. No terminal, suba o ambiente (agora com **três** serviços: broker Kafka, Kafka UI e Schema Registry):

   ```bash
   docker compose up -d
   ```

   Isso pode demorar um pouco mais que nas aulas anteriores, pela imagem adicional do Schema Registry. Confira com `docker compose ps` — devem aparecer `kafka-aula3`, `kafka-ui-aula3` e `schema-registry-aula3`, todos com status `running`.

8. **Confirme que o Schema Registry está respondendo**, antes de seguir para as próximas células. Abra `http://localhost:8081/subjects` no navegador (deve mostrar uma lista vazia `[]`), ou rode no terminal:

   ```bash
   curl http://localhost:8081/subjects
   ```

   Se der erro de conexão, aguarde mais uns 20-30 segundos — o Schema Registry só fica pronto depois do broker Kafka já estar totalmente de pé.

9. (Opcional) Acompanhe visualmente em `http://localhost:8080` (Kafka UI) — esta versão já vem configurada para mostrar também os schemas registrados.

A partir daqui, execute as células em ordem. Ao final da aula:

```bash
docker compose down -v
```


## 1. Configuração inicial e cliente do Schema Registry

O Schema Registry expõe uma API REST simples (porta `8081`). Os endpoints que vamos usar:

| Endpoint | Método | Para quê |
|---|---|---|
| `/subjects/{subject}/versions` | `POST` | Registrar um novo schema para um "subject" (geralmente `{nome-do-tópico}-value`) |
| `/schemas/ids/{id}` | `GET` | Buscar o schema completo a partir do seu ID |
| `/config/{subject}` | `PUT` | Definir o nível de compatibilidade exigido (`BACKWARD`, `FORWARD`, `FULL`, `NONE`) |
| `/compatibility/subjects/{subject}/versions/latest` | `POST` | **Testar** se um schema seria aceito, sem registrá-lo de fato |

Vamos escrever pequenas funções Python em torno dessa API, em vez de usar uma biblioteca cliente pronta — assim fica claro exatamente o que cada operação faz.


In [1]:
import requests
import fastavro
import io
import struct
import json
import time

from kafka import KafkaProducer, KafkaConsumer, KafkaAdminClient
from kafka.admin import NewTopic
from kafka.errors import TopicAlreadyExistsError

BOOTSTRAP_SERVERS = "localhost:9094"
SCHEMA_REGISTRY_URL = "http://localhost:8081"

HEADERS = {"Content-Type": "application/vnd.schemaregistry.v1+json"}


def registrar_schema(subject, schema_dict):
    """Registra um schema para o subject informado. Retorna o schema ID."""
    resp = requests.post(
        f"{SCHEMA_REGISTRY_URL}/subjects/{subject}/versions",
        headers=HEADERS,
        data=json.dumps({"schema": json.dumps(schema_dict)}),
    )
    resp.raise_for_status()
    return resp.json()["id"]


def buscar_schema_por_id(schema_id, cache={}):
    """Busca o schema completo a partir do ID, com um cache local simples
    (na prática, evita uma chamada HTTP para cada mensagem lida)."""
    if schema_id in cache:
        return cache[schema_id]
    resp = requests.get(f"{SCHEMA_REGISTRY_URL}/schemas/ids/{schema_id}")
    resp.raise_for_status()
    schema_dict = json.loads(resp.json()["schema"])
    cache[schema_id] = schema_dict
    return schema_dict


def definir_compatibilidade(subject, nivel):
    """Define o nível de compatibilidade exigido para novos schemas deste subject."""
    resp = requests.put(
        f"{SCHEMA_REGISTRY_URL}/config/{subject}",
        headers=HEADERS,
        data=json.dumps({"compatibility": nivel}),
    )
    resp.raise_for_status()
    print(f"Compatibilidade de '{subject}' definida como {nivel}.")


def testar_compatibilidade(subject, schema_dict):
    """Testa se um schema seria aceito pelo Registry, SEM registrá-lo de fato.
    Retorna True/False."""
    resp = requests.post(
        f"{SCHEMA_REGISTRY_URL}/compatibility/subjects/{subject}/versions/latest",
        headers=HEADERS,
        data=json.dumps({"schema": json.dumps(schema_dict)}),
    )
    resp.raise_for_status()
    return resp.json()["is_compatible"]


print("Cliente do Schema Registry pronto.")


Cliente do Schema Registry pronto.


## 2. Serialização e desserialização Avro (wire format do Confluent)

Quando o Confluent Schema Registry é usado, cada mensagem Avro enviada ao Kafka segue um formato de bytes bem específico (o "wire format"):

```
[1 byte "magic byte", sempre 0]  [4 bytes: schema ID, big-endian]  [dados Avro binários]
```

O **magic byte** identifica que a mensagem usa esse formato; o **schema ID** (não o schema inteiro!) é o que permite que a mensagem seja pequena — o schema completo fica armazenado uma única vez no Registry, e cada mensagem só carrega uma referência de 4 bytes a ele.

Vamos implementar `serializar_avro` e `desserializar_avro` seguindo exatamente esse formato, usando `fastavro` para a parte de codificação Avro em si.


In [2]:
def serializar_avro(schema_dict, schema_id, dados):
    """Codifica 'dados' (um dict) em Avro binário, prefixado pelo wire format do Confluent."""
    buf = io.BytesIO()
    buf.write(struct.pack(">bI", 0, schema_id))  # magic byte (0) + schema id (4 bytes)
    fastavro.schemaless_writer(buf, schema_dict, dados)
    return buf.getvalue()


def desserializar_avro(mensagem_bytes, reader_schema=None):
    """Decodifica uma mensagem no wire format do Confluent.

    'reader_schema' é OPCIONAL: se informado, o fastavro faz a projeção entre o
    schema com que o dado foi ESCRITO (buscado no Registry pelo ID embutido na
    mensagem) e o schema que o CONSUMIDOR quer usar — é exatamente esse
    mecanismo que permite ler dados de uma versão diferente da sua própria.
    """
    magic, schema_id = struct.unpack(">bI", mensagem_bytes[:5])
    assert magic == 0, "Magic byte inesperado — a mensagem não está no wire format do Confluent."

    writer_schema = buscar_schema_por_id(schema_id)
    payload = io.BytesIO(mensagem_bytes[5:])
    return fastavro.schemaless_reader(payload, writer_schema, reader_schema)


print("Funções de serialização Avro prontas.")


Funções de serialização Avro prontas.


## 3. Cenário: múltiplos produtores, múltiplos consumidores, evolução independente

Vamos simular a seguinte situação:

| Papel | Schema usado | Descrição |
|---|---|---|
| **Produtor A** (legado) | v1 — `sensor`, `temperatura` | Continua rodando sem nunca ser atualizado |
| **Produtor B** (novo) | v2 — adiciona `unidade` (campo opcional, com valor padrão) | Serviço recém-implantado, publicando no mesmo tópico |
| **Consumidor 1** (antigo) | v1 | Nunca foi atualizado — não sabe que o campo `unidade` existe |
| **Consumidor 2** (atualizado) | v2 | Já processa `unidade`, e precisa continuar funcionando com dados do Produtor A |
| **Consumidor 3** (agregador) | nenhum campo novo necessário | Só calcula a média de `temperatura` — nem precisa se importar com a evolução |

Nenhum desses papéis é atualizado ao mesmo tempo — é exatamente esse desacoplamento de tempo de deploy que o Schema Registry viabiliza.

### 3.1 Definindo e registrando o schema v1


In [3]:
TOPIC = "leituras-sensores"
SUBJECT = f"{TOPIC}-value"   # convenção padrão do Confluent: {topico}-value

admin = KafkaAdminClient(bootstrap_servers=BOOTSTRAP_SERVERS)
try:
    admin.create_topics([NewTopic(name=TOPIC, num_partitions=1, replication_factor=1)])
    print(f"Tópico '{TOPIC}' criado.")
except TopicAlreadyExistsError:
    print(f"Tópico '{TOPIC}' já existia.")
admin.close()

schema_v1 = {
    "type": "record",
    "name": "Leitura",
    "fields": [
        {"name": "sensor", "type": "string"},
        {"name": "temperatura", "type": "double"},
    ],
}

definir_compatibilidade(SUBJECT, "BACKWARD")
id_schema_v1 = registrar_schema(SUBJECT, schema_v1)
print(f"Schema v1 registrado com ID {id_schema_v1}.")


Tópico 'leituras-sensores' criado.
Compatibilidade de 'leituras-sensores-value' definida como BACKWARD.
Schema v1 registrado com ID 1.


### 3.2 Produtor A publica com o schema v1


In [4]:
def publicar_leitura(schema_dict, schema_id, dados):
    producer = KafkaProducer(bootstrap_servers=BOOTSTRAP_SERVERS)
    mensagem = serializar_avro(schema_dict, schema_id, dados)
    producer.send(TOPIC, value=mensagem)
    producer.flush()
    producer.close()

# Produtor A: 3 leituras, sempre com o schema v1
for i in range(3):
    publicar_leitura(schema_v1, id_schema_v1, {"sensor": f"A-{i}", "temperatura": 20.0 + i})
print("Produtor A publicou 3 leituras com o schema v1.")


Produtor A publicou 3 leituras com o schema v1.


### 3.3 Evoluindo para o schema v2 — adicionando um campo opcional

Vamos adicionar `unidade` como campo **opcional, com valor padrão** — essa é a forma correta de evoluir um schema de forma compatível (`BACKWARD`): um consumidor que já tem o schema v2 consegue ler dados antigos (v1), porque sabe qual valor usar quando o campo não existir no dado original.


In [5]:
schema_v2 = {
    "type": "record",
    "name": "Leitura",
    "fields": [
        {"name": "sensor", "type": "string"},
        {"name": "temperatura", "type": "double"},
        {"name": "unidade", "type": "string", "default": "Celsius"},
    ],
}

# O Registry vai validar a compatibilidade automaticamente ao registrar
id_schema_v2 = registrar_schema(SUBJECT, schema_v2)
print(f"Schema v2 registrado com sucesso, ID {id_schema_v2}. (Se chegou até aqui, o Registry aceitou como compatível.)")


Schema v2 registrado com sucesso, ID 2. (Se chegou até aqui, o Registry aceitou como compatível.)


### 3.4 Produtor B publica com o schema v2, no MESMO tópico, ao mesmo tempo que o Produtor A continuaria publicando


In [6]:
# Produtor B: 3 leituras, com o schema v2 (unidade explícita)
for i in range(3):
    publicar_leitura(schema_v2, id_schema_v2, {"sensor": f"B-{i}", "temperatura": 68.0 + i, "unidade": "Fahrenheit"})
print("Produtor B publicou 3 leituras com o schema v2.")

# Produtor A continua publicando, sem saber que algo mudou
for i in range(3, 6):
    publicar_leitura(schema_v1, id_schema_v1, {"sensor": f"A-{i}", "temperatura": 20.0 + i})
print("Produtor A publicou mais 3 leituras, ainda com o schema v1.")


Produtor B publicou 3 leituras com o schema v2.
Produtor A publicou mais 3 leituras, ainda com o schema v1.


### 3.5 Consumidor 1 (v1) — lê tudo, ignora o campo que não conhece


In [7]:
def consumir_topico(group_id, reader_schema):
    consumer = KafkaConsumer(
        TOPIC,
        bootstrap_servers=BOOTSTRAP_SERVERS,
        group_id=group_id,
        auto_offset_reset="earliest",
        consumer_timeout_ms=5000,
    )
    mensagens = []
    for msg in consumer:
        dados = desserializar_avro(msg.value, reader_schema=reader_schema)
        mensagens.append(dados)
    consumer.close()
    return mensagens

leituras_v1 = consumir_topico("consumidor-1-v1", reader_schema=schema_v1)
for l in leituras_v1:
    print(l)


{'sensor': 'A-0', 'temperatura': 20.0}
{'sensor': 'A-1', 'temperatura': 21.0}
{'sensor': 'A-2', 'temperatura': 22.0}
{'sensor': 'B-0', 'temperatura': 68.0}
{'sensor': 'B-1', 'temperatura': 69.0}
{'sensor': 'B-2', 'temperatura': 70.0}
{'sensor': 'A-3', 'temperatura': 23.0}
{'sensor': 'A-4', 'temperatura': 24.0}
{'sensor': 'A-5', 'temperatura': 25.0}


Repare: as leituras do Produtor B aparecem, mas **sem** o campo `unidade` — o Consumidor 1 nunca pediu esse campo, então ele simplesmente não aparece no resultado. Nada quebrou.

### 3.6 Consumidor 2 (v2) — lê tudo, preenchendo o campo novo quando ele não existir no dado original


In [8]:
leituras_v2 = consumir_topico("consumidor-2-v2", reader_schema=schema_v2)
for l in leituras_v2:
    print(l)


{'sensor': 'A-0', 'temperatura': 20.0, 'unidade': 'Celsius'}
{'sensor': 'A-1', 'temperatura': 21.0, 'unidade': 'Celsius'}
{'sensor': 'A-2', 'temperatura': 22.0, 'unidade': 'Celsius'}
{'sensor': 'B-0', 'temperatura': 68.0, 'unidade': 'Fahrenheit'}
{'sensor': 'B-1', 'temperatura': 69.0, 'unidade': 'Fahrenheit'}
{'sensor': 'B-2', 'temperatura': 70.0, 'unidade': 'Fahrenheit'}
{'sensor': 'A-3', 'temperatura': 23.0, 'unidade': 'Celsius'}
{'sensor': 'A-4', 'temperatura': 24.0, 'unidade': 'Celsius'}
{'sensor': 'A-5', 'temperatura': 25.0, 'unidade': 'Celsius'}


Agora **todas** as leituras têm o campo `unidade` — nas leituras do Produtor A (que nunca enviou esse campo), o Consumidor 2 preencheu automaticamente com o valor padrão (`"Celsius"`), definido no próprio schema v2. Isso é o `fastavro.schemaless_reader` fazendo a projeção entre o `writer_schema` (o que foi usado para escrever aquele dado específico, buscado pelo ID) e o `reader_schema` (o que passamos explicitamente).

### 3.7 Consumidor 3 (agregador) — nem precisa se importar com a evolução


In [9]:
# Este consumidor só quer calcular uma média — passamos reader_schema=None,
# então o fastavro simplesmente usa o próprio writer_schema de cada mensagem
# (a estrutura completa como foi originalmente escrita).
leituras_agregador = consumir_topico("consumidor-3-agregador", reader_schema=None)

temperaturas = [l["temperatura"] for l in leituras_agregador]
print(f"Total de leituras: {len(temperaturas)}")
print(f"Temperatura média (ignorando unidades diferentes, só para ilustrar): {sum(temperaturas) / len(temperaturas):.2f}")


Total de leituras: 9
Temperatura média (ignorando unidades diferentes, só para ilustrar): 38.00


**Nota:** esse último consumidor mistura Celsius e Fahrenheit na média sem perceber — um lembrete de que o Schema Registry garante a *estrutura* dos dados, não o *significado* deles. Um agregador de produção de verdade precisaria checar o campo `unidade` antes de somar valores.


## 4. Evolução incompatível — o Registry rejeitando um schema

Antes de causar uma incompatibilidade de propósito, vale registrar a regra geral (confirmada na documentação oficial do Confluent) para a compatibilidade `BACKWARD`, que definimos para este subject:

- **Remover um campo é permitido** — o consumidor com o schema novo simplesmente ignora esse campo nos dados antigos que ainda o contêm (foi exatamente isso que tornou o schema v2 → v1 seguro, e é por isso que remover `unidade` de volta seria aceito sem problema);
- **Adicionar um campo COM valor padrão é permitido** — foi o que fizemos ao adicionar `unidade`, com `"default": "Celsius"`;
- **Adicionar um campo SEM valor padrão NÃO é permitido** — é exatamente esse o caso que quebra: ao ler dados antigos (que não têm esse campo), o consumidor com o schema novo não teria como preencher o valor, e não existe um "padrão" definido para recorrer.

Vamos demonstrar esse último caso: um schema v3 que adiciona um campo novo `id_leitura` **sem** default.


In [10]:
schema_v3_incompativel = {
    "type": "record",
    "name": "Leitura",
    "fields": [
        {"name": "sensor", "type": "string"},
        {"name": "temperatura", "type": "double"},
        {"name": "unidade", "type": "string", "default": "Celsius"},
        {"name": "id_leitura", "type": "string"},
        # "id_leitura" é um campo NOVO, e não tem "default". Isso quebra a
        # compatibilidade BACKWARD: um consumidor com este schema v3 não
        # teria como preencher "id_leitura" ao ler dados antigos (v1 ou v2),
        # que nunca tiveram esse campo.
    ],
}

try:
    registrar_schema(SUBJECT, schema_v3_incompativel)
    print("Schema v3 foi aceito — isso não deveria acontecer com compatibilidade BACKWARD!")
except requests.exceptions.HTTPError as e:
    print(f"Schema v3 REJEITADO pelo Registry, como esperado. Detalhes: {e.response.text}")


Schema v3 REJEITADO pelo Registry, como esperado. Detalhes: {"error_code":409,"message":"Schema being registered is incompatible with an earlier schema for subject \"leituras-sensores-value\", details: [{errorType:'READER_FIELD_MISSING_DEFAULT_VALUE', description:'The field 'id_leitura' at path '/fields/3' in the new schema has no default value and is missing in the old schema', additionalInfo:'id_leitura'}, {oldSchemaVersion: 2}, {oldSchema: '{\"type\":\"record\",\"name\":\"Leitura\",\"fields\":[{\"name\":\"sensor\",\"type\":\"string\"},{\"name\":\"temperatura\",\"type\":\"double\"},{\"name\":\"unidade\",\"type\":\"string\",\"default\":\"Celsius\"}]}'}, {validateFields: 'false', compatibility: 'BACKWARD'}]"}


Esse é o ponto central do Schema Registry: o erro acontece **no momento de registrar o schema**, de forma controlada e visível — não silenciosamente, meses depois, quando um consumidor em produção quebrar ao processar um dado que já está circulando.

Se você quiser tornar esse mesmo schema v3 compatível, basta adicionar um `"default"` ao campo `id_leitura` (ex.: `"default": ""`) — tente registrar de novo depois de fazer essa mudança, e confirme que passa a ser aceito.


## 5. Atividades Práticas

As três atividades a seguir usam a mesma infraestrutura montada nas seções anteriores. Escreva sua solução na célula de código logo abaixo de cada enunciado.


### Atividade 1 — Validar antes de registrar (evitar o erro em produção)

**Objetivo:** hoje, só descobrimos que um schema é incompatível quando tentamos registrá-lo e o Registry recusa (Seção 4). Isso é *melhor* do que quebrar em produção, mas ainda é tarde — numa esteira de CI/CD, você quer descobrir isso **antes** de sequer tentar o deploy.

**Tarefa:**
1. Escreva uma função `validar_e_registrar(subject, schema_dict)` que:
   - Primeiro chama `testar_compatibilidade(subject, schema_dict)`;
   - Se retornar `True`, chama `registrar_schema(subject, schema_dict)` normalmente e retorna o ID;
   - Se retornar `False`, **não tenta registrar** — em vez disso, levanta uma exceção clara, por exemplo `raise ValueError(f"Schema incompatível com '{subject}', registro abortado.")`.
2. Teste sua função com dois casos:
   - Um schema **compatível** (ex.: adicionar mais um campo opcional com `default` ao schema v2, chamado `precisao_decimal`);
   - Um schema **incompatível** (reaproveite o `schema_v3_incompativel` da Seção 4 — campo novo sem `default` —, ou invente outra mudança do mesmo tipo).

**Pergunta para o relatório:** por que essa checagem prévia (`/compatibility`) é mais adequada para um pipeline de CI/CD do que simplesmente tentar registrar e tratar o erro (como fizemos na Seção 4)?


In [11]:
# Solução da Atividade 1

def validar_e_registrar(subject, schema_dict):
    """Registra o schema somente quando ele é compatível com a versão atual."""
    if not testar_compatibilidade(subject, schema_dict):
        raise ValueError(f"Schema incompatível com '{subject}', registro abortado.")
    return registrar_schema(subject, schema_dict)


# Caso compatível: adiciona um campo com valor padrão.
schema_v3_compativel = {
    "type": "record",
    "name": "Leitura",
    "fields": [
        {"name": "sensor", "type": "string"},
        {"name": "temperatura", "type": "double"},
        {"name": "unidade", "type": "string", "default": "Celsius"},
        {"name": "precisao_decimal", "type": "int", "default": 1},
    ],
}

id_schema_v3 = validar_e_registrar(SUBJECT, schema_v3_compativel)
print(f"Schema compatível registrado com ID {id_schema_v3}.")


# Caso incompatível: a função deve interromper antes de tentar registrar.
try:
    validar_e_registrar(SUBJECT, schema_v3_incompativel)
except ValueError as erro:
    print(erro)


Schema compatível registrado com ID 3.
Schema incompatível com 'leituras-sensores-value', registro abortado.


**Resposta para o relatório:** a checagem prévia pelo endpoint `/compatibility` é mais adequada para CI/CD porque funciona como uma validação sem efeitos colaterais. O pipeline pode falhar antes do deploy e antes de criar uma nova versão no Registry. Isso deixa o erro claro na etapa de teste e evita alterar o estado do ambiente apenas para descobrir que o schema é inválido.

### Atividade 2 — Consumidor genérico multi-schema com contagem por versão

**Objetivo:** os consumidores demonstrados na Seção 3 sempre usaram um `reader_schema` fixo. Nesta atividade, você vai construir um consumidor que **não assume nenhum schema fixo** — ele descobre e conta quantas mensagens vieram de cada schema ID, só olhando o que está no wire format de cada mensagem.

**Tarefa:**
1. Escreva uma função `consumir_com_contagem_por_schema(group_id)` que:
   - Consome todas as mensagens do tópico `leituras-sensores` (use um `group_id` novo, para ler desde o início);
   - Para cada mensagem, **não chame `desserializar_avro` ainda** — primeiro, extraia só o `schema_id` do início da mensagem (os 5 primeiros bytes, com `struct.unpack(">bI", ...)`, igual fazemos dentro de `desserializar_avro`);
   - Mantenha um dicionário `contagem_por_schema` que conta quantas mensagens vieram de cada `schema_id`;
   - Ao final, para cada `schema_id` encontrado, chame `buscar_schema_por_id` para mostrar de qual schema se trata, junto com a contagem.
2. Rode a função e confirme que a contagem bate com o que publicamos nas Seções 3.2 e 3.4 (3 mensagens do schema v1 do Produtor A original + 3 do schema v2 do Produtor B + 3 mais do schema v1 do Produtor A).

**Pergunta para o relatório:** essa técnica (inspecionar o `schema_id` sem desserializar o payload completo) seria útil para depurar rapidamente um tópico com uma mistura inesperada de versões de schema em produção? Por quê?


In [12]:
# Solução da Atividade 2

def consumir_com_contagem_por_schema(group_id):
    consumer = KafkaConsumer(
        TOPIC,
        bootstrap_servers=BOOTSTRAP_SERVERS,
        group_id=group_id,
        auto_offset_reset="earliest",
        consumer_timeout_ms=5000,
    )

    contagem_por_schema = {}
    for msg in consumer:
        magic, schema_id = struct.unpack(">bI", msg.value[:5])
        if magic != 0:
            raise ValueError("Mensagem fora do wire format do Confluent.")
        contagem_por_schema[schema_id] = contagem_por_schema.get(schema_id, 0) + 1

    consumer.close()

    for schema_id, quantidade in sorted(contagem_por_schema.items()):
        schema = buscar_schema_por_id(schema_id)
        campos = [campo["name"] for campo in schema["fields"]]
        print(f"Schema ID {schema_id}: {quantidade} mensagens | campos: {campos}")

    return contagem_por_schema


contagem = consumir_com_contagem_por_schema("atividade-2-contagem-atv07")
print("Contagem final:", contagem)


Schema ID 1: 6 mensagens | campos: ['sensor', 'temperatura']
Schema ID 2: 3 mensagens | campos: ['sensor', 'temperatura', 'unidade']
Contagem final: {1: 6, 2: 3}


**Resposta para o relatório:** sim. O `schema_id` fica nos cinco primeiros bytes e pode ser lido sem desserializar todo o payload. Assim, é possível descobrir rapidamente quais versões estão circulando, a frequência de cada uma e se apareceu uma versão inesperada, com menos processamento e sem precisar conhecer previamente todos os campos.

### Atividade 3 — Quebrando compatibilidade de um jeito diferente (mudança de tipo)

**Objetivo:** a Seção 4 mostrou uma incompatibilidade por **remoção** de campo. Existe outra forma comum de quebrar compatibilidade, mais sutil: **mudar o tipo** de um campo que continua existindo.

**Tarefa:**
1. Defina um schema v4 baseado no schema v2, mas mudando o tipo do campo `temperatura` de `"double"` para `"string"` (mantendo o nome do campo).
2. Use a função `testar_compatibilidade` (da Seção 1, ou a `validar_e_registrar` que você escreveu na Atividade 1) para verificar se esse schema seria aceito.
3. Documente o resultado: foi aceito ou rejeitado? Se puder, use o parâmetro `verbose=true` na chamada HTTP (adicione `?verbose=true` na URL de `/compatibility/...`) para capturar a mensagem de erro exata retornada pelo Registry, e imprima essa mensagem.

**Pergunta para o relatório:** por que mudar `double` para `string` é incompatível, mesmo que o **nome** do campo não tenha mudado? Um consumidor antigo, que espera um número e recebe uma string, teria o mesmo tipo de problema que teria se o campo tivesse sido removido?


In [13]:
# Solução da Atividade 3

schema_v4_tipo_incompativel = {
    "type": "record",
    "name": "Leitura",
    "fields": [
        {"name": "sensor", "type": "string"},
        {"name": "temperatura", "type": "string"},
        {"name": "unidade", "type": "string", "default": "Celsius"},
    ],
}

# verbose=true faz o Registry explicar a incompatibilidade encontrada.
resp = requests.post(
    f"{SCHEMA_REGISTRY_URL}/compatibility/subjects/{SUBJECT}/versions/latest?verbose=true",
    headers=HEADERS,
    data=json.dumps({"schema": json.dumps(schema_v4_tipo_incompativel)}),
)
resp.raise_for_status()
resultado_v4 = resp.json()

print("Schema v4 aceito?", resultado_v4["is_compatible"])
if resultado_v4.get("messages"):
    print("Detalhes retornados pelo Registry:")
    for mensagem in resultado_v4["messages"]:
        print("-", mensagem)


Schema v4 aceito? False
Detalhes retornados pelo Registry:
- {errorType:'TYPE_MISMATCH', description:'The type (path '/fields/1/type') of a field in the new schema does not match with the old schema', additionalInfo:'reader type: STRING not compatible with writer type: DOUBLE'}
- {oldSchemaVersion: 3}
- {oldSchema: '{"type":"record","name":"Leitura","fields":[{"name":"sensor","type":"string"},{"name":"temperatura","type":"double"},{"name":"unidade","type":"string","default":"Celsius"},{"name":"precisao_decimal","type":"int","default":1}]}'}
- {validateFields: 'false', compatibility: 'BACKWARD'}


**Resposta para o relatório:** o schema foi rejeitado. Embora o nome `temperatura` continue igual, o contrato do campo mudou: dados antigos foram gravados como `double`, enquanto o novo leitor exige `string`. O Avro não converte essa mudança automaticamente. Para o consumidor antigo, receber texto onde espera fazer operações numéricas pode causar falha de desserialização ou de processamento. O efeito prático é semelhante ao de um campo ausente: o contrato esperado deixa de ser atendido.

## 6. Extra: como isso fica com `confluent-kafka`

Tudo o que implementamos manualmente nas Seções 1 e 2 (cliente HTTP do Registry, wire format, cache de schemas) já vem embutido na biblioteca oficial `confluent-kafka`, no módulo `confluent_kafka.schema_registry`. Não é necessário instalar ou rodar isso nesta aula — é só para comparação.

```python
# pip install confluent-kafka
from confluent_kafka import Producer, Consumer
from confluent_kafka.schema_registry import SchemaRegistryClient
from confluent_kafka.schema_registry.avro import AvroSerializer, AvroDeserializer
from confluent_kafka.serialization import SerializationContext, MessageField

schema_registry_client = SchemaRegistryClient({"url": "http://localhost:8081"})

avro_serializer = AvroSerializer(
    schema_registry_client,
    json.dumps(schema_v2),   # o mesmo dict de schema que já usamos
)

producer = Producer({"bootstrap.servers": "localhost:9094"})
producer.produce(
    topic=TOPIC,
    value=avro_serializer(
        {"sensor": "C-0", "temperatura": 22.0, "unidade": "Celsius"},
        SerializationContext(TOPIC, MessageField.VALUE),
    ),
)
producer.flush()
```

**O que a biblioteca resolve para você, automaticamente:**
- Registra o schema no Registry (ou reaproveita o ID, se o schema já existir) na primeira vez que o `AvroSerializer` é usado — você não chama `registrar_schema` manualmente;
- Monta e interpreta o wire format (magic byte + schema ID) — não existe `struct.pack`/`struct.unpack` visível no seu código;
- Mantém um cache de schemas internamente (você não escreve o dicionário `cache={}` como fizemos em `buscar_schema_por_id`);
- O `AvroDeserializer`, do lado do consumidor, aceita um `schema_str` opcional para leitura (o equivalente ao nosso `reader_schema`), fazendo a mesma projeção de evolução de schema que vimos com o `fastavro`.

**Por que ainda vale a pena saber fazer manualmente (o que fizemos nas Seções 1-2):** em qualquer situação onde `confluent-kafka` não está disponível (restrições de ambiente, integração com uma linguagem/framework que não tem esse cliente, ou simplesmente para depurar um problema entendendo exatamente os bytes que estão indo para o Kafka), a implementação manual com `requests` + `fastavro` continua funcionando — e agora você sabe exatamente o que o `confluent-kafka` está fazendo por trás da API mais conveniente.

**Nota prática sobre instalação:** o `confluent-kafka` depende de uma biblioteca C (`librdkafka`). Em geral, `pip install confluent-kafka` já baixa um wheel pré-compilado para Windows/Mac/Linux sem precisar compilar nada — mas em redes corporativas restritas (como pode ser o caso do laboratório), vale testar a instalação com antecedência, fora do horário da aula.
